# HPC Workshop — Day 2
## Intel oneAPI, SYCL & HPC Optimization

**Topics:** Intel oneAPI compiler setup, SYCL device discovery, DPC++ vector operations, matrix computation, OpenMP, SIMD, Intel Extension for Scikit-learn (`sklearnex`), and NumPy/BLAS benchmarking.

> Some installation and hardware-specific cells require a Linux environment with appropriate permissions and Intel oneAPI support. Review before running.

# Day 2 — Intel oneAPI, SYCL & HPC Optimization

## 1. Intel oneAPI Installation

Check for an existing oneAPI installation and configure the Intel package repository if required.

> **Caution:** APT installation changes the system and requires sudo, network access, and compatible Linux packages.

In [ ]:
import os

print("[*] Checking for existing Intel oneAPI installation...")
has_oneapi = os.path.exists("/opt/intel/oneapi")

if not has_oneapi:
    print("[*] Configuring Intel Official Package Repository...")
    commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    # Install standalone oneAPI DPC++/C++ compiler and MPI developer headers
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    # We execute setup in background / check readiness
    print("[+] Intel repository commands prepared.")
else:
    print("[+] Intel oneAPI is already installed at /opt/intel/oneapi!")

### 1.1 Install oneAPI compiler and MPI development packages (optional)

In [ ]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

### 1.2 Environment setup script

This minimal script is a placeholder; it does not install oneAPI.

In [ ]:
import os
os.makedirs('/opt/intel/oneapi', exist_ok=True)

# Create a mock/redirecting setvars.sh that exports paths to icpx and system-installed oneAPI tools
setvars_content = """#!/bin/bash
# Automatically exported configurations for oneAPI components
export PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
export PATH=$PATH:$(dirname $(which icpx 2>/dev/null || echo '/usr/bin'))
echo ':: oneAPI environment initialized manually ::'
"""

with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
    f.write(setvars_content)

print('[+] Successfully generated /opt/intel/oneapi/setvars.sh')

### 1.3 Locate `icpx` and configure paths

In [ ]:
import os
import glob

# Find the actual path where icpx resides inside /opt/intel/oneapi
icpx_paths = glob.glob('/opt/intel/oneapi/compiler/*/bin/icpx')
if not icpx_paths:
    icpx_paths = glob.glob('/opt/intel/oneapi/compiler/*/linux/bin/intel64/icpx')

if icpx_paths:
    icpx_path = icpx_paths[0]
    compiler_bin_dir = os.path.dirname(icpx_path)
    print(f"[+] Found icpx compiler at: {icpx_path}")

    # Generate a robust setvars.sh referencing the actual installation paths
    setvars_content = f"""#!/bin/bash
export PATH={compiler_bin_dir}:$PATH
export LD_LIBRARY_PATH=/opt/intel/oneapi/compiler/latest/lib:$LD_LIBRARY_PATH
echo ':: oneAPI Compiler Paths Added ::'
"""
    with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
        f.write(setvars_content)
    print("[+] /opt/intel/oneapi/setvars.sh updated successfully!")
else:
    print("[-] Could not find icpx automatically. Let's list the /opt/intel/oneapi directories to check where it is installed:")
    !find /opt/intel/oneapi -name "icpx"

## 2. SYCL Device Discovery

Enumerate SYCL platforms/devices and inspect compute units, max work-group size, and global memory.

### Extension Task A — Intel Platform Filter
Modify the program to filter platform names containing `Intel` (case-insensitive), and print/count only their devices.

In [ ]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect SYCL platforms, compute devices, and memory capabilities
// ==============================================================================
#include <iostream>
#include <vector>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL oneAPI / SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    // Enumerate all platforms
    auto platforms = sycl::platform::get_platforms();
    std::cout << "[+] Found " << platforms.size() << " SYCL Platform(s):" << std::endl;

    for (size_t p = 0; p < platforms.size(); ++p) {
        std::cout << "\n  Platform [" << p << "]: "
                  << platforms[p].get_info<sycl::info::platform::name>()
                  << " (" << platforms[p].get_info<sycl::info::platform::vendor>() << ")"
                  << std::endl;

        auto devices = platforms[p].get_devices();
        for (size_t d = 0; d < devices.size(); ++d) {
            std::cout << "    └── Device [" << d << "]: "
                      << devices[d].get_info<sycl::info::device::name>() << std::endl;
            std::cout << "        • Compute Units     : "
                      << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
            std::cout << "        • Max Work-Group Size: "
                      << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
            std::cout << "        • Global Memory     : "
                      << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                      << " MB" << std::endl;
        }
    }
#else
    std::cout << "[-] SYCL headers not found. Running Native C++20 Multicore Host Inspector:" << std::endl;
    std::cout << "    • Host Architecture: x86_64 Multicore Processor" << std::endl;
    std::cout << "    • Execution Target : Host CPU SIMD Vector Engine" << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

### 2.1 Compile and run device discovery

In [ ]:
!bash -c "source /opt/intel/oneapi/setvars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query"

## 3. Vector Addition — 10 Million Elements (DPC++/SYCL)

USM arrays and a `parallel_for` compute `C[i] = A[i] + B[i]`, then synchronize, validate, and report execution timing/bandwidth.

In [ ]:
#include <iostream>
#include <chrono>
#include <cmath>
#include <sycl/sycl.hpp>

constexpr size_t N = 10'000'000;

int main() {
    sycl::queue q{sycl::default_selector_v};
    std::cout << "Device: "
              << q.get_device().get_info<sycl::info::device::name>() << '\n';

    float *a = sycl::malloc_shared<float>(N, q);
    float *b = sycl::malloc_shared<float>(N, q);
    float *c = sycl::malloc_shared<float>(N, q);
    if (!a || !b || !c) {
        std::cerr << "USM allocation failed\n";
        if (a) sycl::free(a, q); if (b) sycl::free(b, q); if (c) sycl::free(c, q);
        return 1;
    }

    for (size_t i = 0; i < N; ++i) { a[i] = 1.0f; b[i] = 2.0f; }
    auto start = std::chrono::high_resolution_clock::now();
    q.parallel_for(sycl::range<1>(N), [=](sycl::id<1> i) { c[i] = a[i] + b[i]; });
    q.wait();
    auto end = std::chrono::high_resolution_clock::now();

    bool ok = true;
    for (size_t i = 0; i < 1000; ++i) if (std::abs(c[i] - 3.0f) > 1e-4) { ok = false; break; }
    auto ms = std::chrono::duration<double, std::milli>(end - start).count();
    std::cout << "Kernel time: " << ms << " ms\n"
              << "Estimated bandwidth: " << (3.0 * N * sizeof(float) / (ms * 1e-6) / 1e9) << " GB/s\n"
              << "Verification: " << (ok ? "PASS" : "FAIL") << '\n';
    sycl::free(a, q); sycl::free(b, q); sycl::free(c, q);
}

## 4. Extension Task B — Element-wise Multiplication (5 Million Elements)

Implement `C[i] = A[i] * B[i]` using SYCL `parallel_for` and USM. Starter solution:

In [ ]:
#include <iostream>
#include <sycl/sycl.hpp>

constexpr size_t N = 5'000'000;

int main() {
    sycl::queue q{sycl::default_selector_v};
    float *A = sycl::malloc_shared<float>(N, q);
    float *B = sycl::malloc_shared<float>(N, q);
    float *C = sycl::malloc_shared<float>(N, q);
    if (!A || !B || !C) return 1;

    for (size_t i = 0; i < N; ++i) { A[i] = 2.0f; B[i] = 3.0f; }
    q.parallel_for(sycl::range<1>(N), [=](sycl::id<1> i) {
        C[i] = A[i] * B[i];
    });
    q.wait();
    std::cout << "C[0] = " << C[0] << " (expected 6)\n";
    sycl::free(A, q); sycl::free(B, q); sycl::free(C, q);
}

## 5. Extension Task C — Transposed Matrix Initialization (512 × 512)

Initialize `M[i][j] = i*j`, construct/read its transpose, and verify the mathematical relation `T[j][i] == M[i][j]` with a CPU/OMP loop.

In [ ]:
#include <iostream>
#include <vector>
#include <cassert>

constexpr int N = 512;
int main() {
    std::vector<int> M(N*N), T(N*N);
    #pragma omp parallel for collapse(2)
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j)
            M[i*N+j] = i*j;

    #pragma omp parallel for collapse(2)
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j)
            T[j*N+i] = M[i*N+j];

    bool valid = true;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j)
            if (T[j*N+i] != i*j) valid = false;
    std::cout << "Transpose verification: " << (valid ? "PASS" : "FAIL") << '\n';
}

In [ ]:
# CPU/OpenMP matrix multiplication (the shared code is not SYCL)
#include <iostream>
#include <vector>
#include <chrono>
constexpr size_t D = 1024;
int main() {
    std::vector<float> A(D*D, 1.5f), B(D*D, 2.0f), C(D*D, 0.0f);
    auto start = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for collapse(2)
    for (size_t i=0; i<D; ++i) for (size_t j=0; j<D; ++j) {
        float sum=0;
        for (size_t k=0; k<D; ++k) sum += A[i*D+k]*B[k*D+j];
        C[i*D+j]=sum;
    }
    auto ms=std::chrono::duration<double,std::milli>(
        std::chrono::high_resolution_clock::now()-start).count();
    std::cout<<"Time: "<<ms<<" ms; C[0]="<<C[0]<<" expected "<<D*3.0f<<'\n';
}

In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)

## 7. CPU SIMD / Vectorization Demonstration

Compile with an optimizing compiler and inspect vectorization reports. The pragma is compiler-specific; check compiler support.

In [ ]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect SYCL platforms, compute devices, and memory capabilities
// ==============================================================================
#include <iostream>
#include <vector>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL oneAPI / SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    // Enumerate all platforms
    auto platforms = sycl::platform::get_platforms();
    std::cout << "[+] Found " << platforms.size() << " SYCL Platform(s):" << std::endl;

    for (size_t p = 0; p < platforms.size(); ++p) {
        std::cout << "\n  Platform [" << p << "]: "
                  << platforms[p].get_info<sycl::info::platform::name>()
                  << " (" << platforms[p].get_info<sycl::info::platform::vendor>() << ")"
                  << std::endl;

        auto devices = platforms[p].get_devices();
        for (size_t d = 0; d < devices.size(); ++d) {
            std::cout << "    └── Device [" << d << "]: "
                      << devices[d].get_info<sycl::info::device::name>() << std::endl;
            std::cout << "        • Compute Units     : "
                      << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
            std::cout << "        • Max Work-Group Size: "
                      << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
            std::cout << "        • Global Memory     : "
                      << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                      << " MB" << std::endl;
        }
    }
#else
    std::cout << "[-] SYCL headers not found. Running Native C++20 Multicore Host Inspector:" << std::endl;
    std::cout << "    • Host Architecture: x86_64 Multicore Processor" << std::endl;
    std::cout << "    • Execution Target : Host CPU SIMD Vector Engine" << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

## 8. Intel Extension for scikit-learn (sklearnex)

Install Intel's extension and supporting packages. Restart the kernel if import paths or binary dependencies changed.

In [ ]:
!pip install -U scikit-learn-intelex threadpoolctl py-cpuinfo xgboost 2>&1 | tail -n 6

### 8.1 Ensure `sklearnex` is importable

In [ ]:
import os, sys, site, glob, importlib

def ensure_importable(mod="sklearnex"):
    # Import `mod`; if pip placed it in a directory that is not on sys.path, add that directory.
    try:
        return importlib.import_module(mod)
    except ModuleNotFoundError:
        roots = glob.glob("/usr/local/lib/python3*/*-packages") + site.getsitepackages() + [site.getusersitepackages()]
        for p in roots:
            if os.path.isdir(os.path.join(p, mod)) and p not in sys.path:
                site.addsitedir(p); print("Added to sys.path:", p)
        importlib.invalidate_caches()
        return importlib.import_module(mod)

try:
    sklearnex = ensure_importable("sklearnex")
    from sklearnex import patch_sklearn, unpatch_sklearn
    print("sklearnex import OK")
except Exception as e:
    raise RuntimeError(
        "scikit-learn-intelex could not be imported. Do this: Runtime > Restart session, "
        "then re-run the install cell WITHOUT skipping it and read its output for errors "
        "(python version: %s)." % sys.version.split()[0]) from e

### 8.2 Version, CPU, and timing helpers

In [ ]:
import os, time, warnings, logging
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, sklearnex
warnings.filterwarnings("ignore")
print("scikit-learn:", sklearn.__version__, "| scikit-learn-intelex:", getattr(sklearnex, "__version__", "?"))
print("CPUs:", os.cpu_count())
!lscpu | egrep "Model name|^CPU\(s\)"

def timeit(fn, repeat=2, warmup=0):
    for _ in range(warmup): fn()
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t0)
    return best, out

### 8.3 Global patch, selective patch, and explicit estimator import

In [ ]:
# Method A: global
patch_sklearn()
from sklearn.cluster import KMeans
print("A  ->", KMeans.__module__)          # sklearnex.* when patched

# Method B: selective. Recent releases expect the full dotted names listed by get_patch_names(),
# so we resolve short names against that list (works across versions).
def names_for(*short):
    wanted = {s.lower() for s in short}
    return [n for n in get_patch_names() if n.split(".")[-1].lower() in wanted]

unpatch_sklearn()
print("Patching only:", names_for("KMeans", "PCA"))
patch_sklearn(names_for("KMeans", "PCA"))
from sklearn.cluster import KMeans
from sklearn.svm import SVC
print("B  ->", KMeans.__module__, "|", SVC.__module__, "(SVC stays stock)")

# Method C: explicit import
unpatch_sklearn()
from sklearnex.cluster import KMeans as IntelKMeans
print("C  ->", IntelKMeans.__module__)

### 8.4 List supported accelerated estimators

In [ ]:
from sklearnex import patch_sklearn, unpatch_sklearn, get_patch_names
print("Estimators that can be accelerated:\n", get_patch_names())

### 8.5 Observe accelerated execution and fallback

In [ ]:
// Save as vector_loop_demo.c and compile with a suitable C compiler
#include <stdio.h>
#include <stdlib.h>
#define N 1000000

void compute_simd(float * restrict a, float * restrict b, float * restrict c) {
    #pragma omp simd
    for (int i=0; i<N; ++i) c[i] = a[i]*b[i] + 0.5f;
}
int main() {
    float *a=aligned_alloc(64,N*sizeof(float));
    float *b=aligned_alloc(64,N*sizeof(float));
    float *c=aligned_alloc(64,N*sizeof(float));
    if(!a||!b||!c) return 1;
    for(int i=0;i<N;++i){a[i]=1.0f;b[i]=2.0f;}
    compute_simd(a,b,c);
    printf("SIMD loop completed; c[0]=%f\\n",c[0]);
    free(a);free(b);free(c);
}

## 9. Machine-Learning Benchmark Data Preparation

Create classification, regression, clustering, SVC, and DBSCAN datasets. Some datasets can consume substantial memory.

In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# Scale factor: 1.0 is good for a 2-vCPU Colab VM (~4-6 min total). Use 0.3 for a quick pass, 3+ on a big node.
SCALE = 1.0
rng = np.random.RandomState(42)

Xc, yc = make_classification(int(400_000*SCALE), 40, n_informative=20, random_state=0)
Xr, yr = make_regression(int(400_000*SCALE), 60, noise=5, random_state=0)
Xb, _  = make_blobs(int(600_000*SCALE), 20, centers=8, random_state=0)
Xs, ys = make_classification(int(15_000*SCALE), 30, n_informative=15, random_state=1)     # SVC is O(n^2)
Xd, _  = make_blobs(int(40_000*SCALE), 4, centers=10, cluster_std=0.6, random_state=2)    # DBSCAN
ntest = 20_000
for name in ("Xc","Xr","Xb","Xs","Xd"): print(name, globals()[name].shape)

### 9.1 Forest CoverType dataset (download or synthetic fallback)

In [ ]:
from sklearn.datasets import fetch_covtype
try:
    cov = fetch_covtype(as_frame=False)
    Xall, yall = cov.data.astype(np.float32), cov.target
    src = "Forest CoverType (real)"
except Exception as e:                                     # offline fallback
    Xall, yall = make_classification(581_012, 54, n_informative=30, n_classes=7, random_state=0); Xall = Xall.astype(np.float32); src = "synthetic fallback"
print(src, Xall.shape, "classes:", np.unique(yall))

### 9.2 Compare stock scikit-learn with Intel-optimized estimators

In [ ]:
def end_to_end(optimized: bool, subsample=300_000):
    from sklearnex import patch_sklearn, unpatch_sklearn
    (patch_sklearn if optimized else unpatch_sklearn)()
    from sklearn.model_selection import train_test_split, RandomizedSearchCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import accuracy_score, f1_score, classification_report

    T = {}
    t_all = time.perf_counter()
    X, _, y, _ = train_test_split(Xall, yall, train_size=subsample, stratify=yall, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

    # ---- (1) baseline: linear model
    t0 = time.perf_counter()
    base = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=200))]).fit(Xtr, ytr)
    T["baseline LR fit"] = time.perf_counter() - t0
    acc_base = accuracy_score(yte, base.predict(Xte))

    # ---- (2) kNN with PCA, hyper-parameter search
    pipe = Pipeline([("sc", StandardScaler()), ("pca", PCA(random_state=0)), ("knn", KNeighborsClassifier())])
    grid = {"pca__n_components": [10, 20, 30], "knn__n_neighbors": [3, 5, 9], "knn__weights": ["uniform", "distance"]}
    t0 = time.perf_counter()
    search = RandomizedSearchCV(pipe, grid, n_iter=6, cv=3, scoring="accuracy", random_state=0, n_jobs=1).fit(Xtr, ytr)
    T["PCA+kNN search"] = time.perf_counter() - t0

    t0 = time.perf_counter(); pred = search.predict(Xte); T["inference"] = time.perf_counter() - t0
    T["TOTAL"] = time.perf_counter() - t_all
    return T, dict(baseline_acc=acc_base, best_acc=accuracy_score(yte, pred), macro_f1=f1_score(yte, pred, average="macro"), best_params=search.best_params_)

T0, M0 = end_to_end(False)
T1, M1 = end_to_end(True)
res = pd.DataFrame({"stock_s": T0, "intel_s": T1}); res["speedup"] = res.stock_s / res.intel_s
display(res.round(2))
print("Stock :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M0.items()})
print("Intel :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M1.items()})

## 10. Hardware, SIMD, BLAS & Threading Introspection

In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)

## 11. Python Loops vs NumPy / BLAS Matrix Multiplication

This is a performance demonstration. Pure Python triple loops are intentionally slow; use modest sizes.

In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)

## 12. Workshop Takeaways

- HPC performance depends on parallelism, memory hierarchy, vectorization, and data movement—not only clock speed.
- SYCL abstracts heterogeneous compute devices; USM simplifies shared allocations, but memory placement and device support matter.
- OpenMP parallelizes CPU loops; SIMD vectorization can accelerate independent operations.
- NumPy uses optimized BLAS/LAPACK backends when available.
- Intel `sklearnex` can accelerate supported scikit-learn estimators; unsupported operations may fall back to stock implementations.
- Benchmark fairly: warm up, repeat, report hardware/runtime, and verify correctness.

# Appendix — Repository Source Completeness Check

This appendix preserves the original executable code cells from your friend's repository notebooks.
The organized sections above are explanatory/cleaned for readability; this appendix is the **verbatim source-code reference**.
Some cells are intentionally repeated because the repository itself contains repeated/alternative implementations.
Run only the cells appropriate to your environment (Google Colab/Linux with Intel oneAPI installed).

## Repository notebook: `HPC_ToolKit.ipynb`

In [ ]:
%%writefile vector_loop_demo.c
// Loop Vectorization Demonstration
#include <stdio.h>
#include <stdlib.h>

#define N 1000000

void compute_simd(float* restrict a, float* restrict b, float* restrict c) {
    #pragma clang loop vectorize(enable)
    for (int i = 0; i < N; ++i) {
        c[i] = a[i] * b[i] + 0.5f;
    }
}

int main() {
    float *a = (float*)aligned_alloc(64, N * sizeof(float));
    float *b = (float*)aligned_alloc(64, N * sizeof(float));
    float *c = (float*)aligned_alloc(64, N * sizeof(float));

    for (int i = 0; i < N; ++i) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    compute_simd(a, b, c);
    printf("[+] SIMD Loop Executed Successfully. c[0] = %f\n", c[0]);

    free(a); free(b); free(c);
    return 0;
}
*Vector Loop demo*

## Repository notebook: `Silicon_microarchitecture_&_Cache_Wall_Introspection.ipynb`

In [ ]:
import os

print("[*] Checking for existing Intel oneAPI installation...")
has_oneapi = os.path.exists("/opt/intel/oneapi")

if not has_oneapi:
    print("[*] Configuring Intel Official Package Repository...")
    commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    # Install standalone oneAPI DPC++/C++ compiler and MPI developer headers
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    # We execute setup in background / check readiness
    print("[+] Intel repository commands prepared.")
else:
    print("[+] Intel oneAPI is already installed at /opt/intel/oneapi!")

In [ ]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

In [ ]:
import os
os.makedirs('/opt/intel/oneapi', exist_ok=True)

# Create a mock/redirecting setvars.sh that exports paths to icpx and system-installed oneAPI tools
setvars_content = """#!/bin/bash
# Automatically exported configurations for oneAPI components
export PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
export PATH=$PATH:$(dirname $(which icpx 2>/dev/null || echo '/usr/bin'))
echo ':: oneAPI environment initialized manually ::'
"""

with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
    f.write(setvars_content)

print('[+] Successfully generated /opt/intel/oneapi/setvars.sh')

In [ ]:
!bash -c "source /opt/intel/oneapi/setvars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query"


In [ ]:
import os
import subprocess

# Let's locate the actual vars.sh or icpx path under /opt/intel/oneapi
print("Searching for icpx or setvars in /opt/intel...")
paths = []
for root, dirs, files in os.walk("/opt/intel"):
    for file in files:
        if file in ["setvars.sh", "vars.sh", "icpx"]:
            paths.append(os.path.join(root, file))

print("Found matching files:")
for p in paths:
    print(p)

In [ ]:
# Create a simple sycl_device_query.cpp to compile and test
cpp_code = """#include <sycl/sycl.hpp>
#include <iostream>

int main() {
    for (auto device : sycl::device::get_devices()) {
        std::cout << "Device: " << device.get_info<sycl::info::device::name>() << std::endl;
        std::cout << "  Type: " << (device.is_gpu() ? "GPU" : (device.is_cpu() ? "CPU" : "Host/Other")) << std::endl;
    }
    return 0;
}
"""

with open('sycl_device_query.cpp', 'w') as f:
    f.write(cpp_code)

print("[+] Created sycl_device_query.cpp")

In [ ]:
import subprocess

# Source the correct environment and execute the compiled query binary
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("--- SYCL Device Query Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during execution:")
    print(result.stderr)

In [ ]:
import subprocess

# Compile the SYCL code using the Intel C++ compiler (icpx)
compile_cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- Compilation & Execution Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

In [ ]:
%%writefile vector_mul.cpp
// ==============================================================================
// Program: vector_mul.cpp
// Purpose: Perform parallel element-wise multiplication of two vectors using SYCL
// ==============================================================================
#include <sycl/sycl.hpp>
#include <iostream>
#include <vector>
#include <cmath>

int main() {
    const size_t N = 1000000; // 1 million elements
    std::vector<float> h_a(N, 2.0f);
    std::vector<float> h_b(N, 3.0f);
    std::vector<float> h_c(N, 0.0f);

    try {
        // Initialize SYCL queue with the default selector
        sycl::queue q{sycl::default_selector_v};
        std::cout << "[+] Running on device: "
                  << q.get_device().get_info<sycl::info::device::name>() << std::endl;

        // Create buffers to manage data transfer between host and device
        sycl::buffer<float, 1> buf_a(h_a.data(), sycl::range<1>(N));
        sycl::buffer<float, 1> buf_b(h_b.data(), sycl::range<1>(N));
        sycl::buffer<float, 1> buf_c(h_c.data(), sycl::range<1>(N));

        // Submit command group to the queue
        q.submit([&](sycl::handler& h) {
            // Accessors read and write to buffers
            sycl::accessor acc_a(buf_a, h, sycl::read_only);
            sycl::accessor acc_b(buf_b, h, sycl::read_only);
            sycl::accessor acc_c(buf_c, h, sycl::write_only, sycl::no_init);

            // Execute the kernel in parallel across N work-items
            h.parallel_for(sycl::range<1>(N), [=](sycl::id<1> idx) {
                acc_c[idx] = acc_a[idx] * acc_b[idx];
            });
        }).wait(); // Wait for execution to complete

    } catch (sycl::exception const& e) {
        std::cerr << "[-] SYCL exception caught: " << e.what() << std::endl;
        return 1;
    }

    // Verify the results on the host side
    bool correct = true;
    for (size_t i = 0; i < N; ++i) {
        if (std::abs(h_c[i] - 6.0f) > 1e-5f) {
            correct = false;
            break;
        }
    }

    if (correct) {
        std::cout << "[+] Verification PASSED! Element-wise multiplication computed successfully on " << N << " elements." << std::endl;
    } else {
        std::cout << "[-] Verification FAILED! Incorrect results calculated." << std::endl;
    }

    return 0;
}

In [ ]:
import subprocess
import os

# 1. Reinstall Intel oneAPI if /opt/intel is empty
if not os.path.exists("/opt/intel/oneapi/compiler"):
    print("[*] Re-configuring and installing Intel oneAPI...")
    install_commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    subprocess.run(install_commands, shell=True, executable='/bin/bash')

# 2. Check and run the application using the verified path
vars_path = "/opt/intel/oneapi/compiler/2026.2/env/vars.sh"
if os.path.exists(vars_path):
    print(f"[+] Sourcing environment from: {vars_path}")
    compile_cmd = f"""bash -c 'source {vars_path} && icpx -fsycl vector_mul.cpp -o vector_mul && ./vector_mul'"""
    result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
    print("--- Vector Multiplication Output ---")
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("Error during compilation/execution:")
        print(result.stderr)
else:
    print(f"[-] Error: {vars_path} was not found after installation attempts.")

In [ ]:
import os

# Let's locate the 'sycl/sycl.hpp' header file under /opt/intel/
print("Searching for sycl/sycl.hpp...")
sycl_headers = []
for root, dirs, files in os.walk("/opt/intel"):
    if "sycl.hpp" in files:
        sycl_headers.append(os.path.join(root, "sycl.hpp"))

if sycl_headers:
    print("[+] Found SYCL headers at:")
    for header in sycl_headers:
        print(header)
else:
    print("[-] SYCL header 'sycl.hpp' not found. Please ensure compiler is fully installed.")

In [ ]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect Intel SYCL platforms, compute devices, and memory capabilities
// ==============================================================================
#include <iostream>
#include <vector>
#include <string>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL oneAPI / SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    // Enumerate all platforms and filter for those containing "Intel"
    auto platforms = sycl::platform::get_platforms();
    std::vector<sycl::platform> intel_platforms;

    for (const auto& platform : platforms) {
        std::string platform_name = platform.get_info<sycl::info::platform::name>();
        if (platform_name.find("Intel") != std::string::npos) {
            intel_platforms.push_back(platform);
        }
    }

    std::cout << "[+] Found " << intel_platforms.size() << " Intel SYCL Platform(s):" << std::endl;

    for (size_t p = 0; p < intel_platforms.size(); ++p) {
        std::cout << "\n  Platform [" << p << "]: "
                  << intel_platforms[p].get_info<sycl::info::platform::name>()
                  << " (" << intel_platforms[p].get_info<sycl::info::platform::vendor>() << ")"
                  << std::endl;

        auto devices = intel_platforms[p].get_devices();
        for (size_t d = 0; d < devices.size(); ++d) {
            std::cout << "    └── Device [" << d << "]: "
                      << devices[d].get_info<sycl::info::device::name>() << std::endl;
            std::cout << "        • Compute Units     : "
                      << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
            std::cout << "        • Max Work-Group Size: "
                      << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
            std::cout << "        • Global Memory     : "
                      << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                      << " MB" << std::endl;
        }
    }
#else
    std::cout << "[-] SYCL headers not found. Running Native C++20 Multicore Host Inspector:" << std::endl;
    std::cout << "    • Host Architecture: x86_64 Multicore Processor" << std::endl;
    std::cout << "    • Execution Target : Host CPU SIMD Vector Engine" << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

In [ ]:
%%writefile sycl_gemm.cpp
// ==============================================================================
// Program: sycl_gemm.cpp
// Purpose: 2D Matrix Multiplication in DPC++/SYCL with 2D Range Work-Items
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>

constexpr size_t MATRIX_DIM = 1024; // 1024 x 1024 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "   PARALLEL MATRIX MULTIPLICATION (1024 x 1024)           " << std::endl;
    std::cout << "==========================================================" << std::endl;

    size_t total_elements = MATRIX_DIM * MATRIX_DIM;
    std::vector<float> A(total_elements, 1.5f);
    std::vector<float> B(total_elements, 2.0f);
    std::vector<float> C(total_elements, 0.0f);

    auto start = std::chrono::high_resolution_clock::now();

    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < MATRIX_DIM; ++i) {
        for (size_t j = 0; j < MATRIX_DIM; ++j) {
            float sum = 0.0f;
            for (size_t k = 0; k < MATRIX_DIM; ++k) {
                sum += A[i * MATRIX_DIM + k] * B[k * MATRIX_DIM + j];
            }
            C[i * MATRIX_DIM + j] = sum;
        }
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    double total_flops = 2.0 * MATRIX_DIM * MATRIX_DIM * MATRIX_DIM;
    double gflops = (total_flops * 1e-9) / (duration.count() * 1e-3);

    std::cout << "[+] Matrix Dimensions : " << MATRIX_DIM << " x " << MATRIX_DIM << std::endl;
    std::cout << "[+] Execution Time    : " << duration.count() << " ms" << std::endl;
    std::cout << "[+] Compute Speed     : " << gflops << " GFLOPs" << std::endl;
    std::cout << "[+] Corner Element C[0]: " << C[0] << " (Expected: " << (MATRIX_DIM * 1.5f * 2.0f) << ")" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}

In [ ]:
%%writefile sycl_gemm.cpp
// ==============================================================================
// Program: sycl_gemm.cpp
// Purpose: 2D Matrix Multiplication in DPC++/SYCL with 2D Range Work-Items
// ==============================================================================
#include <sycl/sycl.hpp>
#include <iostream>
#include <vector>
#include <chrono>
#include <cmath>

constexpr size_t MATRIX_DIM = 1024; // 1024 x 1024 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "   PARALLEL SYCL MATRIX MULTIPLICATION (1024 x 1024)      " << std::endl;
    std::cout << "==========================================================" << std::endl;

    size_t total_elements = MATRIX_DIM * MATRIX_DIM;
    std::vector<float> h_A(total_elements, 1.5f);
    std::vector<float> h_B(total_elements, 2.0f);
    std::vector<float> h_C(total_elements, 0.0f);

    auto start = std::chrono::high_resolution_clock::now();

    try {
        // Create SYCL queue with the default selector
        sycl::queue q{sycl::default_selector_v};
        std::cout << "[+] Running on device: "
                  << q.get_device().get_info<sycl::info::device::name>() << std::endl;

        // Define 2D ranges for the buffers
        sycl::range<2> matrix_range(MATRIX_DIM, MATRIX_DIM);

        // Create buffers
        sycl::buffer<float, 2> buf_A(h_A.data(), matrix_range);
        sycl::buffer<float, 2> buf_B(h_B.data(), matrix_range);
        sycl::buffer<float, 2> buf_C(h_C.data(), matrix_range);

        // Submit job to queue
        q.submit([&](sycl::handler& h) {
            // Create accessors
            sycl::accessor acc_A(buf_A, h, sycl::read_only);
            sycl::accessor acc_B(buf_B, h, sycl::read_only);
            sycl::accessor acc_C(buf_C, h, sycl::write_only, sycl::no_init);

            // Execute the 2D parallel kernel
            h.parallel_for(matrix_range, [=](sycl::id<2> idx) {
                size_t row = idx[0];
                size_t col = idx[1];
                float sum = 0.0f;
                for (size_t k = 0; k < MATRIX_DIM; ++k) {
                    sum += acc_A[row][k] * acc_B[k][col];
                }
                acc_C[row][col] = sum;
            });
        }).wait();

    } catch (sycl::exception const& e) {
        std::cerr << "[-] SYCL exception caught: " << e.what() << std::endl;
        return 1;
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    double total_flops = 2.0 * MATRIX_DIM * MATRIX_DIM * MATRIX_DIM;
    double gflops = (total_flops * 1e-9) / (duration.count() * 1e-3);

    std::cout << "[+] Matrix Dimensions : " << MATRIX_DIM << " x " << MATRIX_DIM << std::endl;
    std::cout << "[+] Execution Time    : " << duration.count() << " ms" << std::endl;
    std::cout << "[+] Compute Speed     : " << gflops << " GFLOPs" << std::endl;
    std::cout << "[+] Corner Element C[0]: " << h_C[0] << " (Expected: " << (MATRIX_DIM * 1.5f * 2.0f) << ")" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}

In [ ]:
import subprocess

vars_path = "/opt/intel/oneapi/compiler/2026.2/env/vars.sh"
compile_cmd = f"""bash -c 'source {vars_path} && icpx -fsycl sycl_gemm.cpp -o sycl_gemm && ./sycl_gemm'"""
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- SYCL GEMM Compilation & Execution Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

In [ ]:
!g++ -std=c++20 -O3 -fopenmp -march=native sycl_gemm.cpp -o sycl_gemm && ./sycl_gemm


In [ ]:
%%writefile matrix_transpose.cpp
// ==============================================================================
// Program: matrix_transpose.cpp
// Purpose: Task C: Transposed Matrix Initialization with CPU/OMP Verification
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>
#include <omp.h>
#include <cmath>

constexpr size_t DIM = 512; // 512 x 512 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      TRANSPOSED MATRIX INITIALIZATION & VERIFICATION     " << std::endl;
    std::cout << "==========================================================" << std::endl;

    std::vector<float> A(DIM * DIM, 0.0f);
    std::vector<float> T(DIM * DIM, 0.0f);

    // 1. Initialize original matrix A where A(i, j) = i * j
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            A[i * DIM + j] = static_cast<float>(i * j);
        }
    }

    auto start = std::chrono::high_resolution_clock::now();

    // 2. Compute transpose matrix T where T(j, i) = A(i, j)
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T[j * DIM + i] = A[i * DIM + j];
        }
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    // 3. Verify that the transpose matches our target mathematical values (T(j, i) == j * i)
    bool verified = true;
    #pragma omp parallel for reduction(&& : verified)
    for (size_t j = 0; j < DIM; ++j) {
        for (size_t i = 0; i < DIM; ++i) {
            float expected = static_cast<float>(j * i);
            if (std::abs(T[j * DIM + i] - expected) > 1e-5f) {
                verified = false;
            }
        }
    }

    std::cout << "[+] Matrix Dimensions : " << DIM << " x " << DIM << std::endl;
    std::cout << "[+] Transpose Time    : " << duration.count() << " ms" << std::endl;
    if (verified) {
        std::cout << "[+] Verification PASSED! Transposed matrix matches target mathematical values." << std::endl;
    } else {
        std::cout << "[-] Verification FAILED! Transposed matrix does not match expected values." << std::endl;
    }
    std::cout << "==========================================================" << std::endl;
    return 0;
}

In [ ]:
import subprocess

# Compile the code using GCC with OpenMP enabled and execute it
compile_cmd = "g++ -O3 -fopenmp matrix_transpose.cpp -o matrix_transpose && ./matrix_transpose"
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- CPU/OMP Matrix Transpose Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

In [ ]:
%%writefile matrix_transpose_comparison.cpp
// ==============================================================================
// Program: matrix_transpose_comparison.cpp
// Purpose: Compare sequential vs OpenMP parallel matrix transpose execution times
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>
#include <omp.h>
#include <cmath>

constexpr size_t DIM = 512;

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "    MATRIX TRANSPOSE: SEQUENTIAL VS PARALLEL PERFORMANCE  " << std::endl;
    std::cout << "==========================================================" << std::endl;

    std::vector<float> A(DIM * DIM, 0.0f);
    std::vector<float> T_seq(DIM * DIM, 0.0f);
    std::vector<float> T_par(DIM * DIM, 0.0f);

    // Initialize original matrix A with a simple pattern
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            A[i * DIM + j] = static_cast<float>(i * j);
        }
    }

    // 1. Sequential Transpose (No Parallelization)
    auto start_seq = std::chrono::high_resolution_clock::now();
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T_seq[j * DIM + i] = A[i * DIM + j];
        }
    }
    auto end_seq = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_seq = end_seq - start_seq;

    // 2. Parallel Transpose (With OpenMP Multi-threading)
    auto start_par = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T_par[j * DIM + i] = A[i * DIM + j];
        }
    }
    auto end_par = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_par = end_par - start_par;

    // 3. Verification & Metrics
    bool match = true;
    for (size_t i = 0; i < DIM * DIM; ++i) {
        if (T_seq[i] != T_par[i]) {
            match = false;
            break;
        }
    }

    double speedup = duration_seq.count() / duration_par.count();

    std::cout << "[+] Matrix Size         : " << DIM << " x " << DIM << std::endl;
    std::cout << "[+] Sequential Time     : " << duration_seq.count() << " ms" << std::endl;
    std::cout << "[+] Parallel (OMP) Time : " << duration_par.count() << " ms" << std::endl;
    std::cout << "[+] Measured Speedup    : " << speedup << "x" << std::endl;
    std::cout << "[+] Verification Result : " << (match ? "PASSED (Outputs Match)" : "FAILED (Outputs Differ)") << std::endl;
    std::cout << "==========================================================" << std::endl;

    return 0;
}

In [ ]:
import subprocess

# Compile and run the comparison program using GCC with OpenMP enabled
compile_cmd = "g++ -O3 -fopenmp matrix_transpose_comparison.cpp -o matrix_transpose_comparison && ./matrix_transpose_comparison"
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- Sequential vs. Parallel Transpose Comparison ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

## Repository notebook: `intel_AI_ANALYTICS_TOOLKIT.ipynb`

In [ ]:
!pip install -U scikit-learn-intelex threadpoolctl py-cpuinfo xgboost 2>&1 | tail -n 6

In [ ]:
import os, sys, site, glob, importlib

def ensure_importable(mod="sklearnex"):
    # Import `mod`; if pip placed it in a directory that is not on sys.path, add that directory.
    try:
        return importlib.import_module(mod)
    except ModuleNotFoundError:
        roots = glob.glob("/usr/local/lib/python3*/*-packages") + site.getsitepackages() + [site.getusersitepackages()]
        for p in roots:
            if os.path.isdir(os.path.join(p, mod)) and p not in sys.path:
                site.addsitedir(p); print("Added to sys.path:", p)
        importlib.invalidate_caches()
        return importlib.import_module(mod)

try:
    sklearnex = ensure_importable("sklearnex")
    from sklearnex import patch_sklearn, unpatch_sklearn
    print("sklearnex import OK")
except Exception as e:
    raise RuntimeError(
        "scikit-learn-intelex could not be imported. Do this: Runtime > Restart session, "
        "then re-run the install cell WITHOUT skipping it and read its output for errors "
        "(python version: %s)." % sys.version.split()[0]) from e

In [ ]:
import os, time, warnings, logging
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, sklearnex
warnings.filterwarnings("ignore")
print("scikit-learn:", sklearn.__version__, "| scikit-learn-intelex:", getattr(sklearnex, "__version__", "?"))
print("CPUs:", os.cpu_count())
!lscpu | egrep "Model name|^CPU\(s\)"

def timeit(fn, repeat=2, warmup=0):
    for _ in range(warmup): fn()
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t0)
    return best, out

In [ ]:
from sklearnex import patch_sklearn, unpatch_sklearn, get_patch_names
print("Estimators that can be accelerated:\n", get_patch_names())

In [ ]:
# Method A: global
patch_sklearn()
from sklearn.cluster import KMeans
print("A  ->", KMeans.__module__)          # sklearnex.* when patched

# Method B: selective. Recent releases expect the full dotted names listed by get_patch_names(),
# so we resolve short names against that list (works across versions).
def names_for(*short):
    wanted = {s.lower() for s in short}
    return [n for n in get_patch_names() if n.split(".")[-1].lower() in wanted]

unpatch_sklearn()
print("Patching only:", names_for("KMeans", "PCA"))
patch_sklearn(names_for("KMeans", "PCA"))
from sklearn.cluster import KMeans
from sklearn.svm import SVC
print("B  ->", KMeans.__module__, "|", SVC.__module__, "(SVC stays stock)")

# Method C: explicit import
unpatch_sklearn()
from sklearnex.cluster import KMeans as IntelKMeans
print("C  ->", IntelKMeans.__module__)

In [ ]:
patch_sklearn()
logging.basicConfig(); logging.getLogger("sklearnex").setLevel(logging.INFO)

from sklearn.cluster import KMeans
from sklearn.svm import SVC
X = np.random.rand(20_000, 10)
KMeans(4, n_init=1).fit(X)                                  # supported
SVC(kernel="precomputed").fit(X[:500] @ X[:500].T, np.arange(500) % 2)   # unsupported kernel -> fallback message expected
logging.getLogger("sklearnex").setLevel(logging.WARNING)

In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# Scale factor: 1.0 is good for a 2-vCPU Colab VM (~4-6 min total). Use 0.3 for a quick pass, 3+ on a big node.
SCALE = 1.0
rng = np.random.RandomState(42)

Xc, yc = make_classification(int(400_000*SCALE), 40, n_informative=20, random_state=0)
Xr, yr = make_regression(int(400_000*SCALE), 60, noise=5, random_state=0)
Xb, _  = make_blobs(int(600_000*SCALE), 20, centers=8, random_state=0)
Xs, ys = make_classification(int(15_000*SCALE), 30, n_informative=15, random_state=1)     # SVC is O(n^2)
Xd, _  = make_blobs(int(40_000*SCALE), 4, centers=10, cluster_std=0.6, random_state=2)    # DBSCAN
ntest = 20_000
for name in ("Xc","Xr","Xb","Xs","Xd"): print(name, globals()[name].shape)

In [ ]:
from sklearn.datasets import fetch_covtype
try:
    cov = fetch_covtype(as_frame=False)
    Xall, yall = cov.data.astype(np.float32), cov.target
    src = "Forest CoverType (real)"
except Exception as e:                                     # offline fallback
    Xall, yall = make_classification(581_012, 54, n_informative=30, n_classes=7, random_state=0); Xall = Xall.astype(np.float32); src = "synthetic fallback"
print(src, Xall.shape, "classes:", np.unique(yall))

In [ ]:
def end_to_end(optimized: bool, subsample=300_000):
    from sklearnex import patch_sklearn, unpatch_sklearn
    (patch_sklearn if optimized else unpatch_sklearn)()
    from sklearn.model_selection import train_test_split, RandomizedSearchCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import accuracy_score, f1_score, classification_report

    T = {}
    t_all = time.perf_counter()
    X, _, y, _ = train_test_split(Xall, yall, train_size=subsample, stratify=yall, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

    # ---- (1) baseline: linear model
    t0 = time.perf_counter()
    base = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=200))]).fit(Xtr, ytr)
    T["baseline LR fit"] = time.perf_counter() - t0
    acc_base = accuracy_score(yte, base.predict(Xte))

    # ---- (2) kNN with PCA, hyper-parameter search
    pipe = Pipeline([("sc", StandardScaler()), ("pca", PCA(random_state=0)), ("knn", KNeighborsClassifier())])
    grid = {"pca__n_components": [10, 20, 30], "knn__n_neighbors": [3, 5, 9], "knn__weights": ["uniform", "distance"]}
    t0 = time.perf_counter()
    search = RandomizedSearchCV(pipe, grid, n_iter=6, cv=3, scoring="accuracy", random_state=0, n_jobs=1).fit(Xtr, ytr)
    T["PCA+kNN search"] = time.perf_counter() - t0

    t0 = time.perf_counter(); pred = search.predict(Xte); T["inference"] = time.perf_counter() - t0
    T["TOTAL"] = time.perf_counter() - t_all
    return T, dict(baseline_acc=acc_base, best_acc=accuracy_score(yte, pred), macro_f1=f1_score(yte, pred, average="macro"), best_params=search.best_params_)

T0, M0 = end_to_end(False)
T1, M1 = end_to_end(True)
res = pd.DataFrame({"stock_s": T0, "intel_s": T1}); res["speedup"] = res.stock_s / res.intel_s
display(res.round(2))
print("Stock :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M0.items()})
print("Intel :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M1.items()})

In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)

## Repository notebook: `numpy_vs_numpyblas.ipynb`

In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)